In [1]:
import igraph as ig
import scipy.sparse as sp
import numpy as np
import sknetwork as sn
import CAS

## Format GML data as sparse adjacency

In [5]:
g = ig.Graph.Read_GML("data/eu-core.gml")

In [10]:
adjacency = g.get_adjacency_sparse()
adjacency = adjacency.astype("bool")
adjacency

<Compressed Sparse Row sparse matrix of dtype 'bool'
	with 32770 stored elements and shape (1005, 1005)>

In [13]:
sp.save_npz("data/eu-core_adjacency.npz", adjacency)

In [24]:
labels = np.array(g.vs["gt"], dtype="int32")
labels

array([ 1,  1, 25, ...,  4, 14,  9], shape=(1005,), dtype=int32)

In [25]:
np.save("data/eu-core_labels.npy", labels)

In [27]:
g = ig.Graph.Read_GML("data/as.gml")
g = g.as_undirected()
g.vcount()

23752

In [28]:
adjacency = g.get_adjacency_sparse()
adjacency = adjacency.astype("bool")
adjacency

<Compressed Sparse Row sparse matrix of dtype 'bool'
	with 116832 stored elements and shape (23752, 23752)>

In [29]:
sp.save_npz("data/as_adjacency.npz", adjacency)

In [31]:
vals = np.unique(g.vs["gt"])
id_map = {j: i for i, j in enumerate(vals)}
labels = np.array([id_map[i] for i in g.vs["gt"]])
labels

array([154, 154,   1, ...,  57,   1,   1], shape=(23752,))

In [34]:
np.save("data/as_labels.npy", labels)

In [36]:
g = ig.Graph.Read_GML("data/cora_full.gml")
g = g.as_undirected()
g.vcount()

23166

In [37]:
adjacency = g.get_adjacency_sparse()
adjacency = adjacency.astype("bool")
adjacency

<Compressed Sparse Row sparse matrix of dtype 'bool'
	with 178314 stored elements and shape (23166, 23166)>

In [38]:
sp.save_npz("data/cora_adjacency.npz", adjacency)

In [39]:
vals = np.unique(g.vs["gt"])
id_map = {j: i for i, j in enumerate(vals)}
labels = np.array([id_map[i] for i in g.vs["gt"]])
labels

array([ 0,  0,  0, ..., 57, 15, 45], shape=(23166,))

In [42]:
np.save("data/cora_labels.npy", labels)

## Format SNAP data as sparse

In [6]:
graph = sn.data.from_csv("data/com-amazon.ungraph.txt", comments="#", reindex=True)
adjacency = graph["adjacency"]
adjacency

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 1851744 stored elements and shape (334863, 334863)>

In [ ]:
sp.save_npz("data/amazon_adjacnecy.npz", adjacency)

In [9]:
with open("data/com-amazon.top5000.cmty.txt", "r") as f:
    x = f.readlines()
    labels = sp.dok_matrix((len(x), adjacency.shape[0]), dtype="bool")
    for c, line in enumerate(x):
        nodes = line[:-1].split("\t")
        for n in nodes:
            index = np.argmax(graph["names"] == int(n))
            labels[c, index] = True
labels = labels.tocsr()
labels

<Compressed Sparse Row sparse matrix of dtype 'bool'
	with 67462 stored elements and shape (5000, 334863)>

In [11]:
sp.save_npz("data/amazon_labels2.npz", labels)

In [12]:
graph = sn.data.from_csv("data/com-dblp.ungraph.txt", comments="#", reindex=True)
adjacency = graph["adjacency"]
adjacency

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 2099732 stored elements and shape (317080, 317080)>

In [ ]:
sp.save_npz("data/dblp_adjacnecy.npz", adjacency)

In [13]:
with open("data/com-dblp.top5000.cmty.txt", "r") as f:
    x = f.readlines()
    labels = sp.dok_matrix((len(x), adjacency.shape[0]), dtype="bool")
    for c, line in enumerate(x):
        nodes = line[:-1].split("\t")
        for n in nodes:
            index = np.argmax(graph["names"] == int(n))
            labels[c, index] = True
labels = labels.tocsr()
labels

<Compressed Sparse Row sparse matrix of dtype 'bool'
	with 112228 stored elements and shape (5000, 317080)>

In [14]:
sp.save_npz("data/dblp_labels2.npz", labels)

In [14]:
graph = sn.data.from_csv("data/com-youtube.ungraph.txt", comments="#", reindex=True)
adjacency = graph["adjacency"]
adjacency

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 5975248 stored elements and shape (1134890, 1134890)>

In [15]:
sp.save_npz("data/youtube_adjacency.npz", adjacency)

In [5]:
with open("data/com-youtube.top5000.cmty.txt", "r") as f:
    x = f.readlines()
    labels = sp.dok_matrix((len(x), adjacency.shape[0]), dtype="bool")
    for c, line in enumerate(x):
        nodes = line[:-1].split("\t")
        for n in nodes:
            index = np.argmax(graph["names"] == int(n))
            labels[c, index] = True
labels = labels.tocsr()
labels

<Compressed Sparse Row sparse matrix of dtype 'bool'
	with 72959 stored elements and shape (5000, 1134890)>

In [6]:
sp.save_npz("data/youtube_labels.npz", labels)

# Graph Stats

In [16]:
def load_graph(name):
    adjacency = sp.load_npz(f"data/{name}_adjacency.npz")
    try:
        labels = np.load(f"data/{name}_labels.npy")
        labels_indptr, labels_indices, labels_data = CAS.cas_._labels_array_to_matrix(
            labels
        )
        labels = sp.csr_matrix(
            (labels_data, labels_indices, labels_indptr),
            shape=(len(labels_indptr) - 1, len(labels)),
        )
        labels.data[:] = True
    except FileNotFoundError as e:
        try:
            labels = sp.load_npz(f"data/{name}_labels.npz")
        except FileNotFoundError as e:
            raise ValueError("Can't find labels file as npy or npz :(")
    return adjacency, labels

In [17]:
names = ["football", "eu-core", "cora", "as", "dblp", "amazon", "youtube"]
for name in names:
    adj, lab = load_graph(name)

    print(name)
    print(f"\t |V| = {adj.shape[0]}")
    print(f"\t |E| = {len(adj.data)//2}")
    print(f"\t # Coms = {lab.shape[0]}")
    print(f"\t % Outlier = {np.sum(lab.getnnz(0) == 0) / adj.shape[0] * 100:.0f}")
    n_labels = np.asarray(lab.sum(axis=0))
    eta = np.mean(n_labels[n_labels > 0])
    print(f"\t eta = {eta:.2f}")



football
	 |V| = 115
	 |E| = 613
	 # Coms = 10
	 % Outlier = 10
	 eta = 1.00
eu-core
	 |V| = 1005
	 |E| = 16385
	 # Coms = 42
	 % Outlier = 0
	 eta = 1.00
cora
	 |V| = 23166
	 |E| = 89157
	 # Coms = 70
	 % Outlier = 0
	 eta = 1.00
as
	 |V| = 23752
	 |E| = 58416
	 # Coms = 176
	 % Outlier = 0
	 eta = 1.00
dblp
	 |V| = 317080
	 |E| = 1049866
	 # Coms = 5000
	 % Outlier = 71
	 eta = 1.20
amazon
	 |V| = 334863
	 |E| = 925872
	 # Coms = 5000
	 % Outlier = 95
	 eta = 4.04
youtube
	 |V| = 1134890
	 |E| = 2987624
	 # Coms = 5000
	 % Outlier = 96
	 eta = 1.83
